## 0. Guide

- `2. Multi-view image for each frame` and `5. `don't have to be execute.
- There are 3 kinds of BEV form, currently, we only need to execute `4.0 Define corridor` and `4.3 Per frame BEV` part.

## 1. Load necessary pkl files

In [1]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os

# Load pkl files
train_infos = pickle.load(open("../data/infos/b2d_infos_train.pkl", "rb"))
val_infos = pickle.load(open("../data/infos/b2d_infos_val.pkl", "rb"))
map_infos = pickle.load(open("../data/infos/b2d_map_infos.pkl", "rb"))

print(f"Number of train samples: {len(train_infos)}")
print(f"Number of val samples: {len(val_infos)}")
print(f"Number of towns in map: {len(map_infos)}")

print(type(map_infos))

Number of train samples: 1455
Number of val samples: 840
Number of towns in map: 11
<class 'dict'>


## 2. Multi-view image for each frame

In [ ]:
from tqdm import tqdm

data_root = "bench2drive/"
multi_view_dir = "styleclips/multi-view-images/"

for sample in tqdm(train_infos, desc="Creating multi-view images"):

    scene_dir = os.path.join(multi_view_dir, sample['folder'])
    os.makedirs(scene_dir, exist_ok=True)

    cameras = ['CAM_FRONT', 'CAM_FRONT_LEFT', 'CAM_FRONT_RIGHT',
                   'CAM_BACK', 'CAM_BACK_LEFT', 'CAM_BACK_RIGHT']
    
    output_filename = f"{sample['frame_idx']:05d}.jpg"
    output_path = os.path.join(scene_dir, output_filename)


    # Create the composite image
    fig, axes = plt.subplots(2, 3, figsize=(25, 10))
    axes = axes.flatten()
    
    for idx, cam_name in enumerate(cameras):
        if cam_name in sample['sensors']:
            cam_info = sample['sensors'][cam_name]
            img_path = os.path.join(data_root, cam_info['data_path'])
            # print(f"image_path: {img_path}")
            try:
                img = Image.open(img_path)
                axes[idx].imshow(img)
                axes[idx].set_title(cam_name)
                axes[idx].axis('off')
            except FileNotFoundError:
                axes[idx].text(0.5, 0.5, f'{cam_name}\nFile not found',
                                ha='center', va='center')
                axes[idx].axis('off')
        else:
            axes[idx].text(0.5, 0.5, f'{cam_name}\nNot available',
                            ha='center', va='center')
            axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig(output_path, bbox_inches='tight', dpi=100, format='jpg')
    plt.close(fig)

## 3. Create Clips for QwenVL Analysis

This section demonstrates how to create video clips for QwenVL driving style analysis.

**Sampling Configuration:**
- Clip length: 10 frames (5 seconds at 2Hz)
- Sample interval: 5 frames (from 10Hz original data)
- Sampling pattern: 
  - Clip 0: frames [0, 5, 10, 15, 20, 25, 30, 35, 40, 45]
  - Clip 1: frames [1, 6, 11, 16, 21, 26, 31, 36, 41, 46]
  - etc.

In [2]:
from collections import defaultdict
from typing import List, Dict, Any, Tuple, Optional
import json


class ClipGenerator:
    """Generate clips from Bench2Drive data for QwenVL analysis."""
    
    def __init__(
        self,
        data_infos: List,
        map_infos: Dict,
        data_root: str = "",
        clip_length: int = 10,          # Number of frames per clip
        sample_interval: int = 5,       # Frame interval (original 10Hz, sample at 2Hz -> interval=5)
        max_distance: float = 50.0,     # Max distance for map/object filtering
        multi_view_output_dir: str = "styleclips/multi-view-images",
    ):
        self.data_infos = data_infos
        self.map_infos = map_infos
        self.data_root = data_root
        self.clip_length = clip_length
        self.sample_interval = sample_interval
        self.max_distance = max_distance
        self.multi_view_output_dir = multi_view_output_dir
        
        # Create output directory if it doesn't exist
        os.makedirs(self.multi_view_output_dir, exist_ok=True)
        
        # Group frames by scene
        self.scene_frames = self._group_by_scene()
        
    def _group_by_scene(self) -> Dict[str, List[int]]:
        """Group frame indices by scene/folder."""
        scene_frames = defaultdict(list)
        for idx, info in enumerate(self.data_infos):
            scene_frames[info['folder']].append(idx)
        # print(f"scene frames: {scene_frames}")
        return dict(scene_frames)
    
    def get_clips_for_scene(self, scene_name: str) -> List[List[int]]:
        """
        Get all valid clips for a scene.
        
        Sampling pattern:
        - Clip starting at offset 0: frames [0, 5, 10, 15, 20, 25, 30, 35, 40, 45]
        - Clip starting at offset 1: frames [1, 6, 11, 16, 21, 26, 31, 36, 41, 46]
        - etc.
        """
        frame_indices = self.scene_frames[scene_name] # scene_frames 是把每個 scene 對應的 global frame index 記下來
        num_frames = len(frame_indices) # 這個 scene 有幾個 frames
        required_frames = (self.clip_length - 1) * self.sample_interval + 1 # 每個 clip 橫跨的 index 數
        
        clips = []
        
        # Only iterate over valid starting offsets
        for offset in range(num_frames - required_frames + 1):
            start = offset
            # Create clip with sampled frames
            clip_local_indices = [
                start + i * self.sample_interval 
                for i in range(self.clip_length)
            ]
            # Convert to global indices
            clip_global_indices = [frame_indices[i] for i in clip_local_indices]
            clips.append(clip_global_indices)
        
        return clips # list of list (第二個 list 指的是該 clip 每個 frame 的 global index)
    
    def get_all_clips(self) -> List[Tuple[str, List[int]]]:
        """Get all valid clips from all scenes."""
        all_clips = []
        for scene_name in self.scene_frames.keys():
            clips = self.get_clips_for_scene(scene_name)
            for clip in clips:
                all_clips.append((scene_name, clip))
        return all_clips
    
    def get_frame_info(self, frame_idx: int) -> Dict[str, Any]:
        """Get information for a single frame."""
        sample = self.data_infos[frame_idx]
        # print(sample)
        # Basic ego info
        print(sample['ego_translation'].tolist())
        ego_info = {
            'translation': sample['ego_translation'].tolist() if isinstance(sample['ego_translation'], np.ndarray) else sample['ego_translation'],
            'yaw': ((sample['ego_yaw'])),
            'velocity': ((sample['ego_vel'])),
            'command_near': ((sample['command_near'])),
            'command_far': ((sample['command_far'])) if 'command_far' in sample else None,
        }
       
        # # Object info (filter by valid objects)
        mask = sample['num_points'] != 0
        # print(f"in get frame ingo: mask: {mask}")
        objects = []
        for i, (box, name, obj_id) in enumerate(zip(
            sample['gt_boxes'][mask],
            sample['gt_names'][mask],
            sample['gt_ids'][mask]
        )):
            distance = np.sqrt(box[0]**2 + box[1]**2)
            if distance <= self.max_distance:
                objects.append({
                    'id': int((obj_id)),
                    'class': str(name),
                    'position': [(box[0]), (box[1]), (box[2])],
                    'size': [(box[3]), (box[4]), (box[5])],
                    'yaw': (box[6]),
                    'velocity': [(box[7]), (box[8])],
                    'distance': (distance),
                })
        
        
        return {
            'frame_idx': sample['frame_idx'],
            'ego': ego_info,
            'objects': objects,
        }
    
    def get_trajectory_for_a_clip(self, clip_indices: List[int]) -> Dict :
        past_frames = 2
        future_frames = 7
        total_frames = len(clip_indices)
        cur_idx = clip_indices[2]
        cur_frame = train_infos[cur_idx]

        # Transform matrix from world -> lidar (of current frame)
        world2lidar_cur = np.array(cur_frame['sensors']['LIDAR_TOP']['world2lidar'])
            

        # ========== 1. Collect Ego Trajectory ==========
        ego_positions = np.zeros((total_frames, 2))
        ego_masks = np.zeros(total_frames)
        ego_velocities = np.zeros(total_frames)
        
        for frame_idx, adj_idx in enumerate(clip_indices):
            
            adj_frame = train_infos[adj_idx]
            
            # Get ego position in world coordinate
            ego_pos_world = np.array(adj_frame['ego_translation'])
            
            # Transform to lidar coordinate of current frame
            ego_pos_world_homo = np.concatenate([ego_pos_world, [1.0]])
            ego_pos_lidar = (world2lidar_cur @ ego_pos_world_homo)[:2]
            
            ego_positions[frame_idx] = ego_pos_lidar
            ego_masks[frame_idx] = 1
            ego_velocities[frame_idx] = adj_frame['ego_vel']
            
        # Calculate offset (movement between frames)
        offset_track = ego_positions[1:] - ego_positions[:-1]

        # Accumulate offset for past trajectory (from far to near)      # Numbers from small to large represent points from far to near
        ego_his_trajs = np.zeros((past_frames, 2))
        for j in range(past_frames-1, -1, -1):                          # From 1 ~ 0 (actually just two steps)
            if j == past_frames - 1:                                    # When j = 1
                ego_his_trajs[j] = -offset_track[j]                     # Displacement from current frame to previous frame # 要加負號才對
            else:                                                       # When j = 0
                ego_his_trajs[j] = -(ego_his_trajs[j+1] + offset_track[j]) # Displacement from current frame to two frames ago (accumulated)
        
        # Accumulate offset for future trajectory (from near to far)    # Numbers from small to large represent points from near to far
        ego_fut_trajs = np.zeros((future_frames, 2))
        for j in range(past_frames, past_frames + future_frames):       # From 2 ~ 8
            if j == past_frames:                                        
                ego_fut_trajs[j - past_frames] = offset_track[j]
            else:
                ego_fut_trajs[j - past_frames] = ego_fut_trajs[j - past_frames - 1] + offset_track[j]
        
        ego_fut_masks = ego_masks[past_frames+1:]                   # Not sure what mask is for yet
        
        
        # ========== 2. Collect NPC Trajectory ==========
        # Collect all object IDs across all frames
        object_data = {}  # {object_id: {'positions': [], 'masks': [], 'boxes': [], 'name': str}}

        for frame_idx, adj_idx in enumerate(clip_indices):
        
            adj_frame = train_infos[adj_idx]
            
            # Get transformation from adjacent frame's lidar to current frame's lidar
            world2lidar_adj = np.array(adj_frame['sensors']['LIDAR_TOP']['world2lidar'])
            lidar2world_adj = np.linalg.inv(world2lidar_adj)
            
            # For each object in this frame
            gt_boxes = adj_frame['gt_boxes']
            gt_names = adj_frame['gt_names']
            gt_ids = adj_frame['gt_ids']
            num_points = adj_frame['num_points']
            
            for i in range(len(gt_boxes)):
                obj_id = gt_ids[i]
                obj_name = gt_names[i]
                obj_box = gt_boxes[i]  # [x, y, z, w, l, h, yaw, vx, vy]
                has_points = num_points[i] > 0
                
                # Initialize object entry if not exists
                if obj_id not in object_data:
                    object_data[obj_id] = {
                        'positions': np.zeros((total_frames, 2)),
                        'masks': np.zeros(total_frames),
                        'boxes': np.zeros((total_frames, 9)),
                        'name': obj_name
                    }
                
                # Position in adjacent frame's lidar coordinate
                pos_adj_lidar = np.array([obj_box[0], obj_box[1], obj_box[2], 1.0])
                
                # Transform to world coordinate
                pos_world = lidar2world_adj @ pos_adj_lidar
                
                # Transform to current frame's lidar coordinate
                pos_cur_lidar = world2lidar_cur @ pos_world
                
                object_data[obj_id]['positions'][frame_idx] = pos_cur_lidar[:2]
                object_data[obj_id]['masks'][frame_idx] = 1 if has_points else 0.5  # 0.5 for no points
                object_data[obj_id]['boxes'][frame_idx] = obj_box

    
    def get_map_info_for_frame(self, frame_idx: int) -> Dict[str, Any]:
        """Get map information around the ego vehicle for a single frame."""
        sample = self.data_infos[frame_idx]
        town_name = sample['town_name']
        map_info = self.map_infos[town_name]
        
        world2lidar = np.array(sample['sensors']['LIDAR_TOP']['world2lidar'])
        ego_xy = np.linalg.inv(world2lidar)[0:2, 3]
        
        # Count nearby lanes and triggers
        nearby_lane_types = []
        for i in range(len(map_info['lane_sample_points'])):
            sample_points = map_info['lane_sample_points'][i]
            distance = np.linalg.norm(sample_points[:, 0:2] - ego_xy, axis=-1)
            if distance.min() < self.max_distance:
                nearby_lane_types.append(map_info['lane_types'][i])
        
        nearby_trigger_types = []
        for i in range(len(map_info['trigger_volumes_sample_points'])):
            sample_points = map_info['trigger_volumes_sample_points'][i]
            distance = np.linalg.norm(sample_points[0:2] - ego_xy, axis=-1)
            if distance.min() < self.max_distance:
                nearby_trigger_types.append(map_info['trigger_volumes_types'][i])
        
        return {
            'town_name': town_name,
            'num_lanes': len(nearby_lane_types),
            'num_triggers': len(nearby_trigger_types),
            'lane_types': list(set(nearby_lane_types)),
            'trigger_types': list(set(nearby_trigger_types)),
        }
    
    def _create_multi_view_image(self, sample: Dict, frame_idx: int, scene_name: str) -> str:
        """Create a composite multi-view image from all 6 cameras and save it."""
        cameras = ['CAM_FRONT', 'CAM_FRONT_LEFT', 'CAM_FRONT_RIGHT',
                   'CAM_BACK', 'CAM_BACK_LEFT', 'CAM_BACK_RIGHT']
        
        # Create scene subdirectory
        scene_dir = os.path.join(self.multi_view_output_dir, scene_name)
        os.makedirs(scene_dir, exist_ok=True)
        
        # Generate filename: {frame_idx:05d}.jpg (5 digits with zero padding)
        output_filename = f"{frame_idx:05d}.jpg"
        output_path = os.path.join(scene_dir, output_filename)
        
        # Check if already exists
        if os.path.exists(output_path):
            return output_path
        
        # Create the composite image
        fig, axes = plt.subplots(2, 3, figsize=(25, 10))
        axes = axes.flatten()
        
        for idx, cam_name in enumerate(cameras):
            if cam_name in sample['sensors']:
                cam_info = sample['sensors'][cam_name]
                img_path = os.path.join(self.data_root, cam_info['data_path'])
                print(f"image_path: {img_path}")
                try:
                    img = Image.open(img_path)
                    axes[idx].imshow(img)
                    axes[idx].set_title(cam_name)
                    axes[idx].axis('off')
                except FileNotFoundError:
                    axes[idx].text(0.5, 0.5, f'{cam_name}\nFile not found',
                                   ha='center', va='center')
                    axes[idx].axis('off')
            else:
                axes[idx].text(0.5, 0.5, f'{cam_name}\nNot available',
                               ha='center', va='center')
                axes[idx].axis('off')
        
        plt.tight_layout()
        plt.savefig(output_path, bbox_inches='tight', dpi=100, format='jpg')
        plt.close(fig)
        
        return output_path
    
    def get_image_paths_for_clip(self, clip_indices: List[int]) -> List[str]:
        """Get multi-view composite image file paths for a clip."""
        paths = []
        for frame_idx in clip_indices:
            sample = self.data_infos[frame_idx]
            scene_name = sample['folder']
            
            # Create multi-view composite image and get its path
            multi_view_path = self._create_multi_view_image(sample, frame_idx, scene_name)
            paths.append(multi_view_path)
        
        return paths
    
    def create_clip_data(self, scene_name: str, clip_indices: List[int]) -> Dict[str, Any]:
        """Create complete data dictionary for a clip."""
        # Get image paths
        video_paths = self.get_image_paths_for_clip(clip_indices) 
        
        # Get frame-by-frame info
    
        # frames_info = [self.get_frame_info(idx) for idx in clip_indices]
        frames_info = []
        for idx in clip_indices:
            
            try:
                frames_info.append(self.get_frame_info(idx))
            except Exception as e:
                print(f"[ERROR] get_frame_info failed at idx={idx}, type={type(e).__name__}, msg={e}")
                # 加上關鍵欄位檢查（如果 get_frame_info 裡抓得到 sample）
            
        
        # Get map info (from middle frame)
        middle_idx = clip_indices[len(clip_indices) // 2]
        map_info = self.get_map_info_for_frame(middle_idx)
        
        # Calculate statistics
        ego_velocities = [f['ego']['velocity'] for f in frames_info]
        ego_positions = [f['ego']['translation'] for f in frames_info]
        
        if len(ego_positions) >= 2:
            start_pos = np.array(ego_positions[0][:2])
            end_pos = np.array(ego_positions[-1][:2])
            total_displacement = float(np.linalg.norm(end_pos - start_pos))
        else:
            total_displacement = 0.0
        
        return {
            'scene_name': scene_name,
            'clip_frame_indices': clip_indices,
            'num_frames': len(clip_indices),
            'sample_fps': 2,
            'duration_seconds': (len(clip_indices)) / 2.0,
            'video_paths': video_paths,
            'frames': frames_info,
            'map_info': map_info,
            'statistics': {
                'mean_velocity': float(np.mean(ego_velocities)),
                'max_velocity': float(np.max(ego_velocities)),
                'min_velocity': float(np.min(ego_velocities)),
                'velocity_std': float(np.std(ego_velocities)),
                'total_displacement': total_displacement,
                'total_objects_seen': sum(len(f['objects']) for f in frames_info),
                'unique_object_ids': len(set(obj['id'] for f in frames_info for obj in f['objects'])),
            }
        }
    
    def format_as_qwenvl_message(
        self,
        clip_data: Dict[str, Any],
        prompt: str = "According to the driving record, what do you think about the driving style? conservative, aggressive, or normal? why?",
        include_context: bool = True,
    ) -> Dict[str, Any]:
        """Format clip data as QwenVL message."""
        video_paths = clip_data['video_paths']
        video_paths = [f"bench2drive/{p.lstrip('/')}" for p in video_paths if p is not None]
        
        context_str = ""
        if include_context:
            frames_info = clip_data['frames']
            stats = clip_data['statistics']
            map_info = clip_data['map_info']
            
            context_str = f"""Driving context:
- Duration: {clip_data['duration_seconds']:.1f} seconds
- Average speed: {stats['mean_velocity']:.1f} m/s ({stats['mean_velocity']*3.6:.1f} km/h)
- Speed range: {stats['min_velocity']:.1f} - {stats['max_velocity']:.1f} m/s
- Total displacement: {stats['total_displacement']:.1f} meters
- Objects encountered: {stats['unique_object_ids']} unique objects
- Location: {map_info['town_name']}
- Road features: {', '.join(map_info['lane_types']) if map_info['lane_types'] else 'unknown'}
- Traffic controls: {', '.join(map_info['trigger_types']) if map_info['trigger_types'] else 'none nearby'}
- Detail frames info: {frames_info}

"""
        
        return {
            "role": "user",
            "content": [
                {
                    "type": "video",
                    "video": video_paths,
                    "sample_fps": str(clip_data['sample_fps']),
                },
                {
                    "type": "text",
                    "text": context_str,
                },
                {
                    "type": "text",
                    "text": prompt,
                }
            ],
        }

print("ClipGenerator class defined successfully!")

ClipGenerator class defined successfully!


In [3]:
# Initialize the clip generator
generator = ClipGenerator(
    data_infos=train_infos,
    map_infos=map_infos,
    data_root="bench2drive/",
    clip_length=10,       # 10 frames per clip (5 seconds at 2Hz)
    sample_interval=5,    # Sample every 5 frames (2Hz from 10Hz original)
)

print(f"Number of scenes: {len(generator.scene_frames)}")

# Count total clips
total_clips = sum(len(generator.get_clips_for_scene(s)) for s in generator.scene_frames.keys())
print(f"Total possible clips: {total_clips}")

# Show example for first scene
first_scene = list(generator.scene_frames.keys())[2]
scene_clips = generator.get_clips_for_scene(first_scene)
print(f"\nScene '{first_scene}':")
print(f"  Total frames in scene: {len(generator.scene_frames[first_scene])}")
print(f"  Number of clips: {len(scene_clips)}")
print(f"\n  First 348 clips (frame indices):")
for i, clip in enumerate(scene_clips):
    local_indices = [generator.scene_frames[first_scene].index(idx) for idx in clip]
    print(f"    Clip {i}: local frames {local_indices}")


Number of scenes: 6
Total possible clips: 1185

Scene 'v1/DynamicObjectCrossing_Town02_Route13_Weather6':
  Total frames in scene: 214
  Number of clips: 169

  First 348 clips (frame indices):
    Clip 0: local frames [0, 5, 10, 15, 20, 25, 30, 35, 40, 45]
    Clip 1: local frames [1, 6, 11, 16, 21, 26, 31, 36, 41, 46]
    Clip 2: local frames [2, 7, 12, 17, 22, 27, 32, 37, 42, 47]
    Clip 3: local frames [3, 8, 13, 18, 23, 28, 33, 38, 43, 48]
    Clip 4: local frames [4, 9, 14, 19, 24, 29, 34, 39, 44, 49]
    Clip 5: local frames [5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
    Clip 6: local frames [6, 11, 16, 21, 26, 31, 36, 41, 46, 51]
    Clip 7: local frames [7, 12, 17, 22, 27, 32, 37, 42, 47, 52]
    Clip 8: local frames [8, 13, 18, 23, 28, 33, 38, 43, 48, 53]
    Clip 9: local frames [9, 14, 19, 24, 29, 34, 39, 44, 49, 54]
    Clip 10: local frames [10, 15, 20, 25, 30, 35, 40, 45, 50, 55]
    Clip 11: local frames [11, 16, 21, 26, 31, 36, 41, 46, 51, 56]
    Clip 12: local frames [

## 4. BEV Visualization

### 4.0 Define corridor

In [4]:
def create_trajectory_corridor(positions: np.ndarray, width: float) -> np.ndarray:
    """
    Create a corridor polygon around a trajectory path considering vehicle width.
    
    Args:
        positions: (N, 2) array of trajectory points
        width: Vehicle width
        
    Returns:
        corridor_points: Points forming left and right bounds of the corridor
    """
    if len(positions) < 2:
        return None
    
    half_width = width / 2.0
    left_points = []
    right_points = []
    
    for i in range(len(positions)):
        if i == 0:
            # First point: use direction to next point
            direction = positions[i + 1] - positions[i]
        elif i == len(positions) - 1:
            # Last point: use direction from previous point
            direction = positions[i] - positions[i - 1]
        else:
            # Middle points: average direction
            direction = positions[i + 1] - positions[i - 1]
        
        # Normalize direction
        norm = np.linalg.norm(direction)
        if norm < 1e-6:
            # If points are too close, use previous direction
            if i > 0 and len(left_points) > 0:
                left_points.append(left_points[-1])
                right_points.append(right_points[-1])
            continue
        
        direction = direction / norm
        
        # Perpendicular vector (90 degrees rotation)
        perpendicular = np.array([-direction[1], direction[0]])
        
        # Calculate left and right boundary points
        left_points.append(positions[i] + perpendicular * half_width)
        right_points.append(positions[i] - perpendicular * half_width)
    
    if len(left_points) < 2:
        return None
        
    return np.array(left_points), np.array(right_points)


def create_bev_visualization(
    data_infos: List,
    clip_indices: List[int],
    output_path: str,
    max_distance: float = 50.0,
    figsize: Tuple[int, int] = (12, 12),
    ego_width: float = 2.0,  # Default ego vehicle width
    ego_length: float = 4.5,  # Default ego vehicle length
):
    """
    Create a BEV (Bird's Eye View) visualization for a clip showing:
    - Vehicle positions at the first frame with bounding boxes
    - Trajectories through the clip as corridors with vehicle width
    
    Args:
        data_infos: List of frame information dictionaries
        clip_indices: List of frame indices for the clip
        output_path: Path to save the output image
        max_distance: Maximum distance from ego to include objects
        figsize: Figure size for the plot
        ego_width: Ego vehicle width in meters
        ego_length: Ego vehicle length in meters
    """
    from matplotlib.patches import Rectangle, Polygon
    from matplotlib.transforms import Affine2D
    import matplotlib.patches as mpatches
    
    # Use the first frame as reference coordinate system
    ref_idx = clip_indices[0]
    ref_frame = data_infos[ref_idx]
    world2lidar_ref = np.array(ref_frame['sensors']['LIDAR_TOP']['world2lidar'])
    
    # ========== 1. Collect Ego Trajectory and Yaw ==========
    ego_positions = []
    ego_yaws = []
    for frame_idx in clip_indices:
        frame = data_infos[frame_idx]
        
        # Calculate ego position in reference lidar coordinates
        ego_pos_world = np.array(frame['ego_translation'])
        ego_pos_world_homo = np.concatenate([ego_pos_world, [1.0]])
        ego_pos_lidar = (world2lidar_ref @ ego_pos_world_homo)[:2]
        ego_positions.append(ego_pos_lidar)
        
        # Calculate ego drawing angle in Matplotlib using direction vectors
        v_world = np.array([np.cos(frame['ego_yaw']), np.sin(frame['ego_yaw'])])
        v_ref_lidar = world2lidar_ref[:2, :2] @ v_world
        ego_yaws.append(np.arctan2(v_ref_lidar[1], v_ref_lidar[0]))
        
    ego_positions = np.array(ego_positions)
    
    # ========== 2. Collect NPC Trajectories with Box Info ==========
    object_trajectories = {}  # {obj_id: {'positions': [], 'boxes': [], 'name': str}}
    
    for frame_idx in clip_indices:
        frame = data_infos[frame_idx]
        world2lidar_frame = np.array(frame['sensors']['LIDAR_TOP']['world2lidar'])
        lidar2world_frame = np.linalg.inv(world2lidar_frame)
        
        gt_boxes = frame['gt_boxes']
        gt_names = frame['gt_names']
        gt_ids = frame['gt_ids']
        num_points = frame['num_points']
        
        for i in range(len(gt_boxes)):
            obj_id = gt_ids[i]
            obj_name = gt_names[i]
            obj_box = gt_boxes[i]  # [x, y, z, w, l, h, yaw, vx, vy]
            has_points = num_points[i] > 0
            
            if not has_points:
                continue
                
            # Transform object position to reference frame's lidar coordinate
            pos_frame_lidar = np.array([obj_box[0], obj_box[1], obj_box[2], 1.0])
            pos_world = lidar2world_frame @ pos_frame_lidar
            pos_ref_lidar = world2lidar_ref @ pos_world
            
            # Check distance from ego at first frame
            if np.linalg.norm(pos_ref_lidar[:2]) > max_distance * 2:
                continue
            
            # Calculate object drawing angle in Matplotlib using direction vectors
            v_local_lidar = np.array([np.sin(obj_box[6]), np.cos(obj_box[6])])
            trans_matrix = world2lidar_ref @ lidar2world_frame
            v_ref_lidar = trans_matrix[:2, :2] @ v_local_lidar
            yaw_transformed = np.arctan2(v_ref_lidar[1], v_ref_lidar[0])
            
            if obj_id not in object_trajectories:
                object_trajectories[obj_id] = {
                    'positions': [],
                    'yaws': [],
                    'widths': [],
                    'lengths': [],
                    'name': obj_name,
                    'frame_indices': []
                }
            
            object_trajectories[obj_id]['positions'].append(pos_ref_lidar[:2])
            object_trajectories[obj_id]['yaws'].append(yaw_transformed)
            object_trajectories[obj_id]['widths'].append(obj_box[3])  # w
            object_trajectories[obj_id]['lengths'].append(obj_box[4])  # l
            object_trajectories[obj_id]['frame_indices'].append(frame_idx)
    
    # ========== 3. Create BEV Plot ==========
    fig, ax = plt.subplots(figsize=figsize)
    
    # Color map for different object classes  
    c_blue = "#0F29E9"
    t_Ochre = "#A38209"
    v_green = "#26A41A"
    m_red = "#E01D1D"
    b_purple = "#741BBE"
    p_yellow = "#DAE93E"
    
    # Map class names to colors (similar to b2d_tuto)
    def get_color_for_class(class_name):
        class_name_lower = class_name.lower()
        if 'pedestrian' in class_name_lower or 'walker' in class_name_lower:
            return p_yellow
        elif 'bicycle' in class_name_lower or 'crossbike' in class_name_lower or 'diamondback' in class_name_lower:
            return b_purple
        elif 'motorcycle' in class_name_lower or 'harley' in class_name_lower or 'kawasaki' in class_name_lower or 'vespa' in class_name_lower or 'yamaha' in class_name_lower:
            return m_red
        elif 'bus' in class_name_lower or 'fusorosa' in class_name_lower:
            return "#4BC3EF"
        elif 'truck' in class_name_lower or 'carlamotors' in class_name_lower or 'cybertruck' in class_name_lower:
            return t_Ochre
        elif 'van' in class_name_lower or 'ambulance' in class_name_lower or 'sprinter' in class_name_lower or 'volkswagen.t2' in class_name_lower:
            return v_green
        else:
            return c_blue  # Default for cars
    
    # ========== Plot Ego Trajectory Corridor ==========
    ego_corridor = create_trajectory_corridor(ego_positions, ego_width)
    if ego_corridor is not None:
        left_pts, right_pts = ego_corridor
        # Create filled polygon for ego corridor
        corridor_polygon = np.vstack([left_pts, right_pts[::-1]])
        ax.fill(corridor_polygon[:, 0], corridor_polygon[:, 1], 
                color='green', alpha=0.3, label='Ego Trajectory')
        # Draw corridor outline
        ax.plot(left_pts[:, 0], left_pts[:, 1], 'g-', linewidth=1.5, alpha=0.8)
        ax.plot(right_pts[:, 0], right_pts[:, 1], 'g-', linewidth=1.5, alpha=0.8)
    
    # Draw ego center line
    ax.plot(ego_positions[:, 0], ego_positions[:, 1], 
            color='green', linewidth=2, linestyle='--', alpha=0.8)
    
    # Draw ego bounding box at first frame
    ego_rect = Rectangle((-ego_length/2, -ego_width/2), ego_length, ego_width,
                         linewidth=2, edgecolor='green', facecolor='green', alpha=0.5)
    t = Affine2D().rotate(ego_yaws[0]).translate(ego_positions[0, 0], ego_positions[0, 1]) + ax.transData
    ego_rect.set_transform(t)
    ax.add_patch(ego_rect)
    
    # Mark all ego trajectory points (10 frames)
    for i in range(len(ego_positions)):
        if i == 0:
            # Start position - star marker
            ax.scatter(ego_positions[i, 0], ego_positions[i, 1], 
                       color='green', s=150, marker='*', edgecolors='white', 
                       linewidths=2, zorder=11, label='Ego Start')
        elif i == len(ego_positions) - 1:
            # End position - square marker
            ax.scatter(ego_positions[i, 0], ego_positions[i, 1], 
                       color='darkgreen', s=80, marker='s', edgecolors='white', 
                       linewidths=1, zorder=11)
        else:
            # Intermediate positions - circle markers
            ax.scatter(ego_positions[i, 0], ego_positions[i, 1], 
                       color='green', s=60, marker='o', edgecolors='white', 
                       linewidths=1, zorder=10, alpha=0.8)
    
    # ========== Plot Object Trajectory Corridors ==========
    plotted_types = set()
    for obj_id, obj_data in object_trajectories.items():
        positions = np.array(obj_data['positions'])
        widths = obj_data['widths']
        lengths = obj_data['lengths']
        yaws = obj_data['yaws']
        obj_name = obj_data['name']
        
        if len(positions) < 2:
            continue
        
        # Get color based on class
        color = get_color_for_class(obj_name)
        
        # Use average width for corridor
        avg_width = np.mean(widths)
        avg_length = np.mean(lengths)
        
        # Create trajectory corridor
        obj_corridor = create_trajectory_corridor(positions, avg_width)
        if obj_corridor is not None:
            left_pts, right_pts = obj_corridor
            # Create filled polygon for object corridor
            corridor_polygon = np.vstack([left_pts, right_pts[::-1]])
            
            # Add label only for first object of this type
            label = obj_name.split('.')[-1].capitalize() if obj_name not in plotted_types else None
            ax.fill(corridor_polygon[:, 0], corridor_polygon[:, 1], 
                    color=color, alpha=0.25)
            # Draw corridor outline
            ax.plot(left_pts[:, 0], left_pts[:, 1], color=color, linewidth=1, alpha=0.6)
            ax.plot(right_pts[:, 0], right_pts[:, 1], color=color, linewidth=1, alpha=0.6)
        
        # Draw center line
        ax.plot(positions[:, 0], positions[:, 1], 
                color=color, linewidth=1.5, linestyle='--', alpha=0.7, label=label if obj_corridor is None else None)
        
        # Draw bounding box at first frame
        first_w = widths[0]
        first_l = lengths[0]
        first_yaw = yaws[0]
        rect = Rectangle((-first_l/2, -first_w/2), first_l, first_w,
                         linewidth=1.5, edgecolor=color, facecolor=color, alpha=0.4)
        t = Affine2D().rotate(first_yaw).translate(positions[0, 0], positions[0, 1]) + ax.transData
        rect.set_transform(t)
        ax.add_patch(rect)
        
        # Mark all trajectory points for this object
        for i in range(len(positions)):
            if i == 0:
                # Start position - circle marker (larger)
                ax.scatter(positions[i, 0], positions[i, 1], 
                           color=color, s=60, marker='o', edgecolors='white', 
                           linewidths=1, alpha=0.9, zorder=9)
            elif i == len(positions) - 1:
                # End position - square marker
                ax.scatter(positions[i, 0], positions[i, 1], 
                           color=color, s=40, marker='s', edgecolors='white', 
                           linewidths=1, alpha=0.9, zorder=9)
            else:
                # Intermediate positions - small circle markers
                ax.scatter(positions[i, 0], positions[i, 1], 
                           color=color, s=30, marker='o', edgecolors='white', 
                           linewidths=0.5, alpha=0.7, zorder=8)
        
        plotted_types.add(obj_name)
    
    # ========== 4. Plot Settings ==========
    ax.set_xlim(-max_distance, max_distance)
    ax.set_ylim(-max_distance, max_distance)
    ax.set_xlabel('X (m)', fontsize=12)
    ax.set_ylabel('Y (m)', fontsize=12)
    ax.set_title(f'BEV Trajectory View (with Vehicle Width)\n(Clip: {len(clip_indices)} frames)', fontsize=14)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    
    # Create custom legend
    legend_elements = [
        mpatches.Patch(facecolor='green', alpha=0.5, label='Ego Vehicle'),
        mpatches.Patch(facecolor=c_blue, alpha=0.4, label='Car'),
        mpatches.Patch(facecolor=v_green, alpha=0.4, label='Van'),
        mpatches.Patch(facecolor=t_Ochre, alpha=0.4, label='Truck'),
        mpatches.Patch(facecolor="#4BC3EF", alpha=0.4, label='Bus'),
        mpatches.Patch(facecolor=m_red, alpha=0.4, label='Motorcycle'),
        mpatches.Patch(facecolor=b_purple, alpha=0.4, label='Bicycle'),
        mpatches.Patch(facecolor=p_yellow, alpha=0.4, label='Pedestrian'),
        plt.Line2D([0], [0], marker='*', color='green', label='Start Position', 
                   markersize=12, linestyle='None'),
        plt.Line2D([0], [0], marker='o', color='gray', label='Intermediate', 
                   markersize=6, linestyle='None'),
        plt.Line2D([0], [0], marker='s', color='gray', label='End Position', 
                   markersize=8, linestyle='None'),
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=9)
    
    # Add coordinate reference
    ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.5, alpha=0.5)
    ax.axvline(x=0, color='gray', linestyle='--', linewidth=0.5, alpha=0.5)
    
    # Create output directory if needed
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    # Save figure
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, format='jpg', bbox_inches='tight')
    plt.close(fig)
    
    print(f"BEV visualization saved to: {output_path}")
    return output_path


def create_bev_for_clip(
    generator: ClipGenerator,
    scene_name: str,
    clip_indices: List[int],
    output_dir: str = "styleclips/BEV-images",
    max_distance: float = 50.0,
):
    """
    Create BEV visualization for a specific clip.
    
    Args:
        generator: ClipGenerator instance
        scene_name: Name of the scene
        clip_indices: Frame indices for the clip
        output_dir: Directory to save BEV images
        max_distance: Maximum distance for visualization
    """
    # Create output filename
    start_frame = clip_indices[0]
    end_frame = clip_indices[-1]
    output_filename = f"{scene_name}_clip_{start_frame:05d}_{end_frame:05d}_bev.jpg"
    output_path = os.path.join(output_dir, output_filename)
    
    return create_bev_visualization(
        data_infos=generator.data_infos,
        clip_indices=clip_indices,
        output_path=output_path,
        max_distance=max_distance,
    )

print("BEV visualization functions defined successfully!")

BEV visualization functions defined successfully!


### 4.1 BEV without frame number

In [ ]:
# Create BEV visualization for the first clip
first_scene = list(generator.scene_frames.keys())[0]
first_clip_indices = generator.get_clips_for_scene(first_scene)[0]

print(f"Creating BEV visualization for scene: {first_scene}")
print(f"Clip frame indices: {first_clip_indices}")

# Create the BEV visualization
bev_output_path = create_bev_for_clip(
    generator=generator,
    scene_name=first_scene,
    clip_indices=first_clip_indices,
    output_dir="styleclips/BEV-images",
    max_distance=50.0,
)

# Display the created image
from IPython.display import Image as IPImage, display
display(IPImage(filename=bev_output_path))

### 4.2 BEV with frame number & lane

In [ ]:
# Create BEV visualization with frame number labels
from typing import List, Dict, Tuple

def create_bev_visualization_with_labels(
    data_infos: List,
    map_infos: Dict,
    clip_indices: List[int],
    output_path: str,
    max_distance: float = 50.0,
    figsize: Tuple[int, int] = (14, 14),
    ego_width: float = 2.0,
    ego_length: float = 4.5,
    show_map: bool = True,
):
    """
    Create BEV visualization with frame number labels on each trajectory point.
    Shows trajectory as dashed lines with points, and bounding box only at first frame.
    Optionally shows lane lines and traffic triggers from map data.
    """
    from matplotlib.patches import Rectangle
    from matplotlib.transforms import Affine2D
    import matplotlib.patches as mpatches
    
    # Use the first frame as reference coordinate system
    ref_idx = clip_indices[0]
    ref_frame = data_infos[ref_idx]
    world2lidar_ref = np.array(ref_frame['sensors']['LIDAR_TOP']['world2lidar'])
    
    # Collect Ego Trajectory and Yaw
    ego_positions = []
    ego_yaws = []
    for frame_idx in clip_indices:
        frame = data_infos[frame_idx]
        
        # Calculate ego position in reference lidar coordinates
        ego_pos_world = np.array(frame['ego_translation'])
        ego_pos_world_homo = np.concatenate([ego_pos_world, [1.0]])
        ego_pos_lidar = (world2lidar_ref @ ego_pos_world_homo)[:2]
        ego_positions.append(ego_pos_lidar)
        
        # Calculate ego drawing angle in Matplotlib using direction vectors
        v_world = np.array([np.cos(frame['ego_yaw']), np.sin(frame['ego_yaw'])])
        v_ref_lidar = world2lidar_ref[:2, :2] @ v_world
        ego_yaws.append(np.arctan2(v_ref_lidar[1], v_ref_lidar[0]))
        
    ego_positions = np.array(ego_positions)
    
    # Collect NPC Trajectories
    object_trajectories = {}
    
    for clip_frame_num, frame_idx in enumerate(clip_indices):
        frame = data_infos[frame_idx]
        world2lidar_frame = np.array(frame['sensors']['LIDAR_TOP']['world2lidar'])
        lidar2world_frame = np.linalg.inv(world2lidar_frame)
        
        gt_boxes = frame['gt_boxes']
        gt_names = frame['gt_names']
        gt_ids = frame['gt_ids']
        num_points = frame['num_points']
        
        for i in range(len(gt_boxes)):
            obj_id = gt_ids[i]
            obj_name = gt_names[i]
            obj_box = gt_boxes[i]
            has_points = num_points[i] > 0
            
            if not has_points:
                continue
                
            pos_frame_lidar = np.array([obj_box[0], obj_box[1], obj_box[2], 1.0])
            pos_world = lidar2world_frame @ pos_frame_lidar
            pos_ref_lidar = world2lidar_ref @ pos_world
            
            if np.linalg.norm(pos_ref_lidar[:2]) > max_distance * 2:
                continue
            
            # obj_box[6] is in the current frame's lidar coordinates
            # Calculate object drawing angle in Matplotlib using direction vectors
            v_local_lidar = np.array([np.sin(obj_box[6]), np.cos(obj_box[6])])
            trans_matrix = world2lidar_ref @ lidar2world_frame
            v_ref_lidar = trans_matrix[:2, :2] @ v_local_lidar
            yaw_transformed = np.arctan2(v_ref_lidar[1], v_ref_lidar[0])
            
            if obj_id not in object_trajectories:
                object_trajectories[obj_id] = {
                    'positions': [],
                    'yaws': [],
                    'widths': [],
                    'lengths': [],
                    'name': obj_name,
                    'clip_frame_nums': []
                }
            
            object_trajectories[obj_id]['positions'].append(pos_ref_lidar[:2])
            object_trajectories[obj_id]['yaws'].append(yaw_transformed)
            object_trajectories[obj_id]['widths'].append(obj_box[3])
            object_trajectories[obj_id]['lengths'].append(obj_box[4])
            object_trajectories[obj_id]['clip_frame_nums'].append(clip_frame_num)
    
    # ========== Collect Map Data (Lane Lines & Triggers) ==========
    nearby_lanes = []
    nearby_lane_types = []
    nearby_triggers = []
    nearby_trigger_types = []
    
    if show_map:
        # Get map info for the reference frame
        town_name = ref_frame['town_name']
        map_info = map_infos[town_name]
        
        # Get ego position in world coordinates
        ego_xy = np.linalg.inv(world2lidar_ref)[0:2, 3]
        
        # Collect nearby lane lines
        for idx in range(len(map_info['lane_sample_points'])):
            sample_points = map_info['lane_sample_points'][idx]
            distance = np.linalg.norm(sample_points[:, 0:2] - ego_xy, axis=-1)
            
            if distance.min() < max_distance:
                # Transform to lidar coordinate
                lane_points_world = map_info['lane_points'][idx]
                lane_points_homo = np.concatenate([
                    lane_points_world,
                    np.ones((len(lane_points_world), 1))
                ], axis=-1)
                lane_points_lidar = (world2lidar_ref @ lane_points_homo.T).T[:, :2]
                
                nearby_lanes.append(lane_points_lidar)
                nearby_lane_types.append(map_info['lane_types'][idx])
        
        # Collect nearby trigger volumes (traffic lights, stop signs)
        for idx in range(len(map_info['trigger_volumes_sample_points'])):
            sample_points = map_info['trigger_volumes_sample_points'][idx]
            distance = np.linalg.norm(sample_points[0:2] - ego_xy, axis=-1)
            
            if distance.min() < max_distance:
                trigger_points_world = map_info['trigger_volumes_points'][idx]
                trigger_points_homo = np.concatenate([
                    trigger_points_world,
                    np.ones((len(trigger_points_world), 1))
                ], axis=-1)
                trigger_points_lidar = (world2lidar_ref @ trigger_points_homo.T).T[:, :2]
                
                nearby_triggers.append(trigger_points_lidar)
                nearby_trigger_types.append(map_info['trigger_volumes_types'][idx])
    
    # Create BEV Plot
    fig, ax = plt.subplots(figsize=figsize)
    
    # Set white background
    ax.set_facecolor('white')
    fig.patch.set_facecolor('white')
    
    # ========== Draw Map Elements First (as background) ==========
    if show_map:
        # Lane color and linestyle mapping
        lane_color_map = {
            'Broken': '#888888',      # gray
            'Solid': "#000000",       # black
            'SolidSolid': '#CC8800',  # dark orange (double solid)
            'Center': '#FF5500',      # orange (center line)
            'NONE': "#000000"
        }
        lane_linestyle_map = {
            'Broken': '--',   # dashed
            'Solid': '-',
            'SolidSolid': '-',
            'Center': '-',
            'NONE': '-',
        }
        trigger_color_map = {
            'TrafficLight': '#008800',  # dark green
            'StopSign': '#CC0000',      # dark red
        }
        
        # Draw lane lines
        lane_legend_added = set()
        for lane_pts, lane_type in zip(nearby_lanes, nearby_lane_types):
            color = lane_color_map.get(lane_type, '#888888')
            linestyle = lane_linestyle_map.get(lane_type, '-')
            label = lane_type if lane_type not in lane_legend_added else None
            ax.plot(lane_pts[:, 0], lane_pts[:, 1],
                    color=color, linewidth=1.2, alpha=0.6, linestyle=linestyle, 
                    label=label, zorder=1)
            lane_legend_added.add(lane_type)
        
        # Draw trigger volumes (traffic lights / stop signs)
        trigger_legend_added = set()
        for trig_pts, trig_type in zip(nearby_triggers, nearby_trigger_types):
            color = trigger_color_map.get(trig_type, '#AA00AA')
            label = trig_type if trig_type not in trigger_legend_added else None
            ax.fill(trig_pts[:, 0], trig_pts[:, 1],
                    color=color, alpha=0.25, label=label, zorder=2)
            ax.plot(np.append(trig_pts[:, 0], trig_pts[0, 0]),
                    np.append(trig_pts[:, 1], trig_pts[0, 1]),
                    color=color, linewidth=1.0, alpha=0.6, zorder=2)
            trigger_legend_added.add(trig_type)
    
    # Color definitions for vehicles
    c_blue = "#0F29E9"
    t_Ochre = "#A38209"
    v_green = "#26A41A"
    m_red = "#E01D1D"
    b_purple = "#741BBE"
    p_yellow = "#DAE93E"
    
    def get_color_for_class(class_name):
        class_name_lower = class_name.lower()
        if 'pedestrian' in class_name_lower or 'walker' in class_name_lower:
            return p_yellow
        elif 'bicycle' in class_name_lower or 'crossbike' in class_name_lower or 'diamondback' in class_name_lower:
            return b_purple
        elif 'motorcycle' in class_name_lower or 'harley' in class_name_lower or 'kawasaki' in class_name_lower or 'vespa' in class_name_lower or 'yamaha' in class_name_lower:
            return m_red
        elif 'bus' in class_name_lower or 'fusorosa' in class_name_lower:
            return "#4BC3EF"
        elif 'truck' in class_name_lower or 'carlamotors' in class_name_lower or 'cybertruck' in class_name_lower:
            return t_Ochre
        elif 'van' in class_name_lower or 'ambulance' in class_name_lower or 'sprinter' in class_name_lower or 'volkswagen.t2' in class_name_lower:
            return v_green
        else:
            return c_blue
    
    # Draw ego trajectory as dashed line
    ax.plot(ego_positions[:, 0], ego_positions[:, 1], 
            color='green', linewidth=2, linestyle='--', alpha=0.8, zorder=5)
    
    # Draw ego bounding box at first frame
    ego_rect = Rectangle((-ego_length/2, -ego_width/2), ego_length, ego_width,
                         linewidth=2, edgecolor='green', facecolor='green', alpha=0.5)
    t = Affine2D().rotate(ego_yaws[0]).translate(ego_positions[0, 0], ego_positions[0, 1]) + ax.transData
    ego_rect.set_transform(t)
    ax.add_patch(ego_rect)
    
    # Mark all ego trajectory points with frame numbers
    for i in range(len(ego_positions)):
        if i == 0:
            ax.scatter(ego_positions[i, 0], ego_positions[i, 1], 
                       color='green', s=150, marker='*', edgecolors='white', 
                       linewidths=2, zorder=11)
        elif i == len(ego_positions) - 1:
            ax.scatter(ego_positions[i, 0], ego_positions[i, 1], 
                       color='darkgreen', s=80, marker='s', edgecolors='white', 
                       linewidths=1, zorder=11)
        else:
            ax.scatter(ego_positions[i, 0], ego_positions[i, 1], 
                       color='green', s=60, marker='o', edgecolors='white', 
                       linewidths=1, zorder=10, alpha=0.8)
        
        # Add frame number label for ego
        ax.annotate(f'{i}', 
                    xy=(ego_positions[i, 0], ego_positions[i, 1]),
                    xytext=(3, 3), textcoords='offset points',
                    fontsize=8, fontweight='bold', color='darkgreen',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7, edgecolor='green'),
                    zorder=12)
    
    # Plot Object Trajectories with frame labels
    plotted_types = set()
    for obj_id, obj_data in object_trajectories.items():
        positions = np.array(obj_data['positions'])
        widths = obj_data['widths']
        lengths = obj_data['lengths']
        yaws = obj_data['yaws']
        obj_name = obj_data['name']
        clip_frame_nums = obj_data['clip_frame_nums']
        
        # Only plot objects that appear in at least 3 frames
        if len(positions) < 3:
            continue
        
        color = get_color_for_class(obj_name)
        
        # Draw trajectory as dashed line (only if more than 1 point)
        if len(positions) >= 2:
            ax.plot(positions[:, 0], positions[:, 1], 
                    color=color, linewidth=1.5, linestyle='--', alpha=0.7, zorder=5)
        
        # Draw bounding box only at first frame where this object appears
        first_w = widths[0]
        first_l = lengths[0]
        first_yaw = yaws[0]
        rect = Rectangle((-first_l/2, -first_w/2), first_l, first_w,
                         linewidth=1.5, edgecolor=color, facecolor=color, alpha=0.4)
        t = Affine2D().rotate(first_yaw).translate(positions[0, 0], positions[0, 1]) + ax.transData
        rect.set_transform(t)
        ax.add_patch(rect)
        
        # Mark all trajectory points with frame numbers
        for i in range(len(positions)):
            frame_num = clip_frame_nums[i]
            if i == 0:
                ax.scatter(positions[i, 0], positions[i, 1], 
                           color=color, s=60, marker='o', edgecolors='white', 
                           linewidths=1, alpha=0.9, zorder=9)
            elif i == len(positions) - 1:
                ax.scatter(positions[i, 0], positions[i, 1], 
                           color=color, s=40, marker='s', edgecolors='white', 
                           linewidths=1, alpha=0.9, zorder=9)
            else:
                ax.scatter(positions[i, 0], positions[i, 1], 
                           color=color, s=30, marker='o', edgecolors='white', 
                           linewidths=0.5, alpha=0.7, zorder=8)
            
            # Add frame number label
            ax.annotate(f'{frame_num}', 
                        xy=(positions[i, 0], positions[i, 1]),
                        xytext=(3, 3), textcoords='offset points',
                        fontsize=7, fontweight='bold', color=color,
                        bbox=dict(boxstyle='round,pad=0.15', facecolor='white', alpha=0.6, 
                                  edgecolor=color, linewidth=0.5), zorder=10)
        
        plotted_types.add(obj_name)
    
    # Plot Settings
    ax.set_xlim(-max_distance, max_distance)
    ax.set_ylim(-max_distance, max_distance)
    ax.set_xlabel('X (m)', fontsize=12, color='black')
    ax.set_ylabel('Y (m)', fontsize=12, color='black')
    title = f'BEV Trajectory View with Frame Labels'
    if show_map:
        title += ' + Map'
    title += f'\n(Clip: {len(clip_indices)} frames, Frame 0-{len(clip_indices)-1})'
    ax.set_title(title, fontsize=14, color='black')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3, color='gray')
    
    # Update tick colors for white background
    ax.tick_params(colors='black')
    for spine in ax.spines.values():
        spine.set_edgecolor('black')
    
    # Legend - Build dynamically based on what's shown
    legend_elements = []
    
    # Map elements (if shown)
    if show_map and len(nearby_lanes) > 0:
        # Add a separator line
        legend_elements.append(plt.Line2D([0], [0], color='white', linewidth=0, label='── Map ──'))
        # Lane types
        lane_types_shown = set(nearby_lane_types)
        if 'Broken' in lane_types_shown:
            legend_elements.append(plt.Line2D([0], [0], color='#888888', linewidth=2, 
                                             linestyle='--', label='Broken Lane'))
        if 'Solid' in lane_types_shown:
            legend_elements.append(plt.Line2D([0], [0], color='#000000', linewidth=2, 
                                             linestyle='-', label='Solid Lane'))
        if 'SolidSolid' in lane_types_shown:
            legend_elements.append(plt.Line2D([0], [0], color='#CC8800', linewidth=2, 
                                             linestyle='-', label='Double Solid'))
        if 'Center' in lane_types_shown:
            legend_elements.append(plt.Line2D([0], [0], color='#FF5500', linewidth=2, 
                                             linestyle='-', label='Center Line'))
    
    if show_map and len(nearby_triggers) > 0:
        # Trigger types
        trigger_types_shown = set(nearby_trigger_types)
        if 'TrafficLight' in trigger_types_shown:
            legend_elements.append(mpatches.Patch(facecolor='#008800', alpha=0.25, 
                                                 edgecolor='#008800', label='Traffic Light'))
        if 'StopSign' in trigger_types_shown:
            legend_elements.append(mpatches.Patch(facecolor='#CC0000', alpha=0.25, 
                                                 edgecolor='#CC0000', label='Stop Sign'))
    
    # Vehicle elements
    if len(legend_elements) > 0:
        legend_elements.append(plt.Line2D([0], [0], color='white', linewidth=0, label='── Vehicles ──'))
    
    legend_elements.extend([
        mpatches.Patch(facecolor='green', alpha=0.5, label='Ego Vehicle'),
        mpatches.Patch(facecolor=c_blue, alpha=0.4, label='Car'),
        mpatches.Patch(facecolor=v_green, alpha=0.4, label='Van'),
        mpatches.Patch(facecolor=t_Ochre, alpha=0.4, label='Truck'),
        mpatches.Patch(facecolor="#4BC3EF", alpha=0.4, label='Bus'),
        mpatches.Patch(facecolor=m_red, alpha=0.4, label='Motorcycle'),
        mpatches.Patch(facecolor=b_purple, alpha=0.4, label='Bicycle'),
        mpatches.Patch(facecolor=p_yellow, alpha=0.4, label='Pedestrian'),
    ])
    
    # Frame markers
    legend_elements.extend([
        plt.Line2D([0], [0], color='white', linewidth=0, label='── Frames ──'),
        plt.Line2D([0], [0], marker='*', color='green', label='Frame 0 (Start)', 
                   markersize=12, linestyle='None'),
        plt.Line2D([0], [0], marker='s', color='gray', label='Last Frame (End)', 
                   markersize=8, linestyle='None'),
    ])
    
    ax.legend(handles=legend_elements, loc='upper right', fontsize=9,
              facecolor='white', labelcolor='black', framealpha=0.9, edgecolor='black')
    
    ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.5, alpha=0.5)
    ax.axvline(x=0, color='gray', linestyle='--', linewidth=0.5, alpha=0.5)
    
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, format='jpg', bbox_inches='tight')
    plt.close(fig)
    
    print(f"BEV visualization with labels saved to: {output_path}")
    return output_path
    
    print(f"BEV visualization with labels saved to: {output_path}")
    return output_path

In [ ]:
from IPython.display import Image as IPImage, display
# Create visualization with frame labels
first_scene = list(generator.scene_frames.keys())[0]
first_clip_indices = generator.get_clips_for_scene(first_scene)[0]

print(f"Creating BEV with frame labels for scene: {first_scene}")
print(f"Clip frame indices: {first_clip_indices}")
print(f"Frame numbers in clip: 0-9 (total 10 frames)")

bev_labeled_path = create_bev_visualization_with_labels(
    data_infos=generator.data_infos,
    map_infos=map_infos,
    clip_indices=first_clip_indices,
    output_path="styleclips/BEV-images/labeled_" + first_scene + "_bev.jpg",
    max_distance=50.0,
    show_map=True,
)

display(IPImage(filename=bev_labeled_path))


### 4.3 Per frame BEV

In [8]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.transforms import Affine2D
import matplotlib.patches as mpatches
from typing import List, Dict, Tuple
from IPython.display import Image as IPImage, display

def create_per_frame_bev(
    data_infos: List,
    map_infos: Dict,
    clip_indices: List[int],
    output_dir: str,
    scene_name: str,
    max_distance: float = 50.0,
    figsize: Tuple[int, int] = (14, 14),
    ego_width: float = 2.0,
    ego_length: float = 4.5,
    show_map: bool = True,
):
    
    os.makedirs(output_dir, exist_ok=True)
    
    ref_idx = clip_indices[0]
    ref_frame = data_infos[ref_idx]
    world2lidar_ref = np.array(ref_frame['sensors']['LIDAR_TOP']['world2lidar'])
    
    ego_positions = []
    ego_yaws = []
    for frame_idx in clip_indices:
        frame = data_infos[frame_idx]
        ego_pos_world = np.array(frame['ego_translation'])
        ego_pos_world_homo = np.concatenate([ego_pos_world, [1.0]])
        ego_pos_lidar = (world2lidar_ref @ ego_pos_world_homo)[:2]
        ego_positions.append(ego_pos_lidar)
        
        v_world = np.array([np.cos(frame['ego_yaw']), np.sin(frame['ego_yaw'])])
        v_ref_lidar = world2lidar_ref[:2, :2] @ v_world
        ego_yaws.append(np.arctan2(v_ref_lidar[1], v_ref_lidar[0]))
        
    ego_positions = np.array(ego_positions)
    
    object_trajectories = {}
    for clip_frame_num, frame_idx in enumerate(clip_indices):
        frame = data_infos[frame_idx]
        world2lidar_frame = np.array(frame['sensors']['LIDAR_TOP']['world2lidar'])
        lidar2world_frame = np.linalg.inv(world2lidar_frame)
        
        gt_boxes = frame['gt_boxes']
        gt_names = frame['gt_names']
        gt_ids = frame['gt_ids']
        num_points = frame['num_points']
        
        for i in range(len(gt_boxes)):
            obj_id = gt_ids[i]
            obj_name = gt_names[i]
            obj_box = gt_boxes[i]
            has_points = num_points[i] > 0
            
            if not has_points:
                continue
                
            pos_frame_lidar = np.array([obj_box[0], obj_box[1], obj_box[2], 1.0])
            pos_world = lidar2world_frame @ pos_frame_lidar
            pos_ref_lidar = world2lidar_ref @ pos_world
            
            if np.linalg.norm(pos_ref_lidar[:2]) > max_distance * 2:
                continue
            
            v_local_lidar = np.array([np.sin(obj_box[6]), np.cos(obj_box[6])])
            trans_matrix = world2lidar_ref @ lidar2world_frame
            v_ref_lidar = trans_matrix[:2, :2] @ v_local_lidar
            yaw_transformed = np.arctan2(v_ref_lidar[1], v_ref_lidar[0])
            
            if obj_id not in object_trajectories:
                object_trajectories[obj_id] = {
                    'positions': [], 'yaws': [], 'widths': [], 'lengths': [],
                    'name': obj_name, 'clip_frame_nums': []
                }
            
            object_trajectories[obj_id]['positions'].append(pos_ref_lidar[:2])
            object_trajectories[obj_id]['yaws'].append(yaw_transformed)
            object_trajectories[obj_id]['widths'].append(obj_box[3])
            object_trajectories[obj_id]['lengths'].append(obj_box[4])
            object_trajectories[obj_id]['clip_frame_nums'].append(clip_frame_num)
            
    nearby_lanes = []
    nearby_lane_types = []
    nearby_triggers = []
    nearby_trigger_types = []
    
    if show_map:
        town_name = ref_frame['town_name']
        map_info = map_infos[town_name]
        ego_xy = np.linalg.inv(world2lidar_ref)[0:2, 3]
        
        for idx in range(len(map_info['lane_sample_points'])):
            sample_points = map_info['lane_sample_points'][idx]
            if np.linalg.norm(sample_points[:, 0:2] - ego_xy, axis=-1).min() < max_distance:
                lane_points_world = map_info['lane_points'][idx]
                lane_points_homo = np.concatenate([lane_points_world, np.ones((len(lane_points_world), 1))], axis=-1)
                nearby_lanes.append((world2lidar_ref @ lane_points_homo.T).T[:, :2])
                nearby_lane_types.append(map_info['lane_types'][idx])
                
        for idx in range(len(map_info['trigger_volumes_sample_points'])):
            sample_points = map_info['trigger_volumes_sample_points'][idx]
            if np.linalg.norm(sample_points[0:2] - ego_xy, axis=-1).min() < max_distance:
                trigger_points_world = map_info['trigger_volumes_points'][idx]
                trigger_points_homo = np.concatenate([trigger_points_world, np.ones((len(trigger_points_world), 1))], axis=-1)
                nearby_triggers.append((world2lidar_ref @ trigger_points_homo.T).T[:, :2])
                nearby_trigger_types.append(map_info['trigger_volumes_types'][idx])
                
    def get_color_for_class(class_name):
        class_name_lower = class_name.lower()
        if 'pedestrian' in class_name_lower or 'walker' in class_name_lower: return "#DAE93E"
        elif 'bicycle' in class_name_lower or 'crossbike' in class_name_lower or 'diamondback' in class_name_lower: return "#741BBE"
        elif 'motorcycle' in class_name_lower or 'harley' in class_name_lower or 'kawasaki' in class_name_lower or 'vespa' in class_name_lower or 'yamaha' in class_name_lower: return "#E01D1D"
        elif 'bus' in class_name_lower or 'fusorosa' in class_name_lower: return "#4BC3EF"
        elif 'truck' in class_name_lower or 'carlamotors' in class_name_lower or 'cybertruck' in class_name_lower: return "#A38209"
        elif 'van' in class_name_lower or 'ambulance' in class_name_lower or 'sprinter' in class_name_lower or 'volkswagen.t2' in class_name_lower: return "#26A41A"
        else: return "#0F29E9"

    output_paths = []
    
    for current_i in range(len(clip_indices)):
        fig, ax = plt.subplots(figsize=figsize)
        ax.set_facecolor('white')
        fig.patch.set_facecolor('white')
        
        if show_map:
            lane_color_map = {'Broken': '#888888', 'Solid': '#000000', 'SolidSolid': '#CC8800', 'Center': '#FF5500', 'NONE': '#000000'}
            lane_linestyle_map = {'Broken': '--', 'Solid': '-', 'SolidSolid': '-', 'Center': '-', 'NONE': '-'}
            trigger_color_map = {'TrafficLight': '#008800', 'StopSign': '#CC0000'}
            
            for lane_pts, lane_type in zip(nearby_lanes, nearby_lane_types):
                color = lane_color_map.get(lane_type, '#888888')
                linestyle = lane_linestyle_map.get(lane_type, '-')
                ax.plot(lane_pts[:, 0], lane_pts[:, 1], color=color, linewidth=1.2, alpha=0.6, linestyle=linestyle, zorder=1)
                
            for trig_pts, trig_type in zip(nearby_triggers, nearby_trigger_types):
                color = trigger_color_map.get(trig_type, '#AA00AA')
                ax.fill(trig_pts[:, 0], trig_pts[:, 1], color=color, alpha=0.25, zorder=2)
                ax.plot(np.append(trig_pts[:, 0], trig_pts[0, 0]), np.append(trig_pts[:, 1], trig_pts[0, 1]), color=color, linewidth=1.0, alpha=0.6, zorder=2)

        if current_i > 0:
            ax.plot(ego_positions[:current_i+1, 0], ego_positions[:current_i+1, 1], 
                    color='green', linewidth=2, linestyle='--', alpha=0.8, zorder=5)
            
        ego_rect = Rectangle((-ego_length/2, -ego_width/2), ego_length, ego_width,
                             linewidth=2, edgecolor='green', facecolor='green', alpha=0.5)
        t = Affine2D().rotate(ego_yaws[current_i]).translate(ego_positions[current_i, 0], ego_positions[current_i, 1]) + ax.transData
        ego_rect.set_transform(t)
        ax.add_patch(ego_rect)
        
        ax.scatter(ego_positions[current_i, 0], ego_positions[current_i, 1], 
                   color='green', s=100, marker='*', edgecolors='white', 
                   linewidths=2, zorder=11)

        for obj_id, obj_data in object_trajectories.items():
            positions = np.array(obj_data['positions'])
            clip_frame_nums = obj_data['clip_frame_nums']
            
            if current_i not in clip_frame_nums:
                continue
            
            valid_indices = [idx for idx, f_num in enumerate(clip_frame_nums) if f_num <= current_i]
            if not valid_indices:
                continue
                
            cur_obj_positions = positions[valid_indices]
            color = get_color_for_class(obj_data['name'])
            
            if len(cur_obj_positions) >= 2:
                ax.plot(cur_obj_positions[:, 0], cur_obj_positions[:, 1], 
                        color=color, linewidth=1.5, linestyle='--', alpha=0.7, zorder=5)
            
            idx_at_current = clip_frame_nums.index(current_i)
            w = obj_data['widths'][idx_at_current]
            l = obj_data['lengths'][idx_at_current]
            yaw = obj_data['yaws'][idx_at_current]
            pos = positions[idx_at_current]
            
            rect = Rectangle((-l/2, -w/2), l, w,
                             linewidth=1.5, edgecolor=color, facecolor=color, alpha=0.4)
            t = Affine2D().rotate(yaw).translate(pos[0], pos[1]) + ax.transData
            rect.set_transform(t)
            ax.add_patch(rect)
            
            ax.scatter(pos[0], pos[1], color=color, s=60, marker='o', edgecolors='white', linewidths=1, alpha=0.9, zorder=9)

        ax.set_xlim(-max_distance, max_distance)
        ax.set_ylim(-max_distance, max_distance)
        ax.set_xlabel('X (m)', fontsize=12, color='black')
        ax.set_ylabel('Y (m)', fontsize=12, color='black')
        ax.set_title(f'Per Frame BEV - Frame {current_i}', fontsize=14, color='black')
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.3, color='gray')

        # === Legend Setup ===
        legend_elements = []
        if show_map and len(nearby_lanes) > 0:
            legend_elements.append(plt.Line2D([0], [0], color='white', linewidth=0, label='── Map ──'))
            lane_types_shown = set(nearby_lane_types)
            if 'Broken' in lane_types_shown: legend_elements.append(plt.Line2D([0], [0], color='#888888', linewidth=2, linestyle='--', label='Broken Lane'))
            if 'Solid' in lane_types_shown: legend_elements.append(plt.Line2D([0], [0], color='#000000', linewidth=2, linestyle='-', label='Solid Lane'))
            if 'SolidSolid' in lane_types_shown: legend_elements.append(plt.Line2D([0], [0], color='#CC8800', linewidth=2, linestyle='-', label='Double Solid'))
            if 'Center' in lane_types_shown: legend_elements.append(plt.Line2D([0], [0], color='#FF5500', linewidth=2, linestyle='-', label='Center Line'))
        
        if show_map and len(nearby_triggers) > 0:
            trigger_types_shown = set(nearby_trigger_types)
            if 'TrafficLight' in trigger_types_shown: legend_elements.append(mpatches.Patch(facecolor='#008800', alpha=0.25, edgecolor='#008800', label='Traffic Light'))
            if 'StopSign' in trigger_types_shown: legend_elements.append(mpatches.Patch(facecolor='#CC0000', alpha=0.25, edgecolor='#CC0000', label='Stop Sign'))
        
        if len(legend_elements) > 0:
            legend_elements.append(plt.Line2D([0], [0], color='white', linewidth=0, label='── Vehicles ──'))
        
        legend_elements.extend([
            mpatches.Patch(facecolor='green', alpha=0.5, label='Ego Vehicle'),
            mpatches.Patch(facecolor="#0F29E9", alpha=0.4, label='Car'),
            mpatches.Patch(facecolor="#26A41A", alpha=0.4, label='Van'),
            mpatches.Patch(facecolor="#A38209", alpha=0.4, label='Truck'),
            mpatches.Patch(facecolor="#4BC3EF", alpha=0.4, label='Bus'),
            mpatches.Patch(facecolor="#E01D1D", alpha=0.4, label='Motorcycle'),
            mpatches.Patch(facecolor="#741BBE", alpha=0.4, label='Bicycle'),
            mpatches.Patch(facecolor="#DAE93E", alpha=0.4, label='Pedestrian'),
        ])
        
        ax.legend(handles=legend_elements, loc='upper right', fontsize=9,
                  facecolor='white', labelcolor='black', framealpha=0.9, edgecolor='black')
        # ====================
        
        # Save
        scene_name_safe = scene_name.replace("/", "_")
        output_path = os.path.join(output_dir, f"{scene_name_safe}_frame_{current_i:02d}.jpg")
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        plt.tight_layout()
        plt.savefig(output_path, dpi=150, format='jpg', bbox_inches='tight')
        plt.close(fig)
        
        output_paths.append(output_path)
    
    return output_paths

In [9]:
target_scene_num = 0

for target_clip_num in range(0, 1, 1):
    target_scene = list(generator.scene_frames.keys())[target_scene_num]
    target_clip_indices = generator.get_clips_for_scene(target_scene)[target_clip_num]

    target_scene_safe = target_scene.replace("/", "_").replace("v1_", "")
    output_dir = os.path.join("styleclips", "BEV-images", "per_frame", f"{target_scene_safe}", f"clip_{target_clip_num}")

    print(f"Creating 10 per-frame BEVs for scene: {target_scene}")
    generated_files = create_per_frame_bev(
        data_infos=generator.data_infos,
        map_infos=map_infos,
        clip_indices=target_clip_indices,
        output_dir=output_dir,
        scene_name=target_scene,
        max_distance=50.0,
        show_map=True,
    )

    print(f"Saved {len(generated_files)} images to {output_dir}")

Creating 10 per-frame BEVs for scene: v1/ConstructionObstacle_Town05_Route68_Weather8
Saved 10 images to styleclips/BEV-images/per_frame/ConstructionObstacle_Town05_Route68_Weather8/clip_0


## 5.

In [ ]:
# Create and examine a single clip
first_scene = list(generator.scene_frames.keys())[0]
first_clip_indices = generator.get_clips_for_scene(first_scene)[0]

print(f"Creating clip from scene: {first_scene}")
print(f"Frame indices: {first_clip_indices}")

# Create clip data
clip_data = generator.create_clip_data(first_scene, first_clip_indices)

print(f"\n{'='*60}")
print("Clip Data Summary")
print(f"{'='*60}")
print(f"Scene: {clip_data['scene_name']}")
print(f"Number of frames: {clip_data['num_frames']}")
print(f"Duration: {clip_data['duration_seconds']} seconds")
print(f"Sample FPS: {clip_data['sample_fps']}")

print(f"\n--- Statistics ---")
for key, value in clip_data['statistics'].items():
    if isinstance(value, float):
        print(f"  {key}: {value:.2f}")
    else:
        print(f"  {key}: {value}")

print(f"\n--- Map Info ---")
for key, value in clip_data['map_info'].items():
    print(f"  {key}: {value}")

print(f"\n--- Frame-by-frame Ego Info ---")
for i, frame in enumerate(clip_data['frames']):
    print(frame)
    # ego = frame['ego']
    # print(f"  Frame {i}: vel={ego['velocity']:.2f} m/s, yaw={ego['yaw']:.2f}, objects={len(frame['objects'])}")


In [ ]:
# Format as QwenVL message
qwenvl_message = generator.format_as_qwenvl_message(
    clip_data,
    prompt="According to the driving record, what do you think about the driving style? conservative, aggressive, or normal? why?",
    include_context=True,
)

print("QwenVL Message Format:")
print("="*60)
print(json.dumps(qwenvl_message, indent=2))


In [ ]:
# Generate all clips and save to JSON
# This may take a while depending on dataset size

from tqdm import tqdm

def create_qwenvl_dataset(
    generator: ClipGenerator,
    output_path: str,
    max_clips: Optional[int] = None,
    prompt: str = "According to the driving record, what do you think about the driving style? conservative, aggressive, or normal? why?",
    include_context: bool = True,
):
    """Create full QwenVL dataset from all clips."""
    all_clips = generator.get_all_clips()
    
    if max_clips is not None:
        all_clips = all_clips[:max_clips]
    
    messages = []
    clip_metadata = []
    
    for scene_name, clip_indices in tqdm(all_clips, desc="Creating clips"):
        clip_data = generator.create_clip_data(scene_name, clip_indices)
        
        message = generator.format_as_qwenvl_message(
            clip_data, 
            prompt=prompt,
            include_context=include_context,
        )
        
        messages.append(message)
        clip_metadata.append({
            'scene_name': scene_name,
            'frame_indices': clip_indices,
            'statistics': clip_data['statistics'],
            'map_info': clip_data['map_info'],
        })
    
    # Save to file
    output_data = {
        'messages': messages,
        'metadata': clip_metadata,
        'config': {
            'clip_length': generator.clip_length,
            'sample_interval': generator.sample_interval,
            'sample_fps': 2,
            'max_distance': generator.max_distance,
        }
    }
    
    with open(output_path, 'w') as f:
        json.dump(output_data, f, indent=2)
    
    print(f"Saved {len(messages)} clips to {output_path}")
    return messages, clip_metadata

# Create dataset with first 50 clips as demo
messages, metadata = create_qwenvl_dataset(
    generator,
    output_path="styleclips/qwenvl_clips_demo.json",
    max_clips=50,
    include_context=True,
)

print(f"\nGenerated {len(messages)} QwenVL messages")


## Usage Notes

### Clip Structure

Each clip contains:
- **10 frames** sampled at 2Hz (5 seconds total)
- **Video paths**: File paths to CAM_FRONT images
- **Frame info**: Ego vehicle state and detected objects per frame
- **Map info**: Lane types, traffic controls in the area
- **Statistics**: Velocity stats, displacement, object counts

### QwenVL Message Format

```python
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "video",
                "video": [
                    "file:///path/to/frame1.jpg",
                    "file:///path/to/frame2.jpg",
                    ...
                ],
                "sample_fps": "2",
            },
            {
                "type": "text", 
                "text": "Driving context:\n- Duration: 4.5 seconds\n..."
            },
        ],
    }
]
```

### Sampling Pattern

With `clip_length=10` and `sample_interval=5`:
- Clip offset 0: frames [0, 5, 10, 15, 20, 25, 30, 35, 40, 45]
- Clip offset 1: frames [1, 6, 11, 16, 21, 26, 31, 36, 41, 46]
- Clip offset 2: frames [2, 7, 12, 17, 22, 27, 32, 37, 42, 47]
- Clip offset 3: frames [3, 8, 13, 18, 23, 28, 33, 38, 43, 48]
- Clip offset 4: frames [4, 9, 14, 19, 24, 29, 34, 39, 44, 49]

### Generate Full Dataset

```python
# For full dataset (may take time)
messages, metadata = create_qwenvl_dataset(
    generator,
    output_path="styleclips/qwenvl_clips_full.json",
    max_clips=None,  # Process all clips
    include_context=True,
)
```